# KR-BERT로 영화 리뷰 감성 분류하기

서울대 컴퓨터언어학 연구실이 공개한 한국어 언어모델 **KR-BERT**를 내려받아, 네이버 영화 리뷰가 긍정인지 부정인지 맞히도록 학습시킵니다.

**실행 전에 꼭 할 일**: 메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택하세요. GPU 없이도 돌아가지만 수십 배 느립니다.

전체 실행 시간은 GPU 기준 5~10분 정도입니다.

## 1. 라이브러리 설치

- `transformers`: 공개된 언어모델을 내려받아 쓰는 라이브러리
- `datasets`: 데이터를 모델 입력 형태로 다루는 라이브러리

In [ ]:
!pip install -q transformers datasets accelerate

## 2. 데이터 불러오기

NSMC(Naver Sentiment Movie Corpus)는 영화 리뷰 20만 개에 긍정(1)/부정(0) 라벨이 붙은 공개 데이터입니다. KR-BERT 논문도 감성 분석 성능을 이 데이터로 측정했습니다.

처음에는 빨리 결과를 보기 위해 학습 2만 개, 평가 5천 개만 뽑아 씁니다. 익숙해지면 `N_TRAIN`을 늘려 보세요.

In [ ]:
import pandas as pd

BASE = "https://raw.githubusercontent.com/e9t/nsmc/master/"
N_TRAIN, N_TEST = 20000, 5000

# quoting=3: 리뷰 본문에 들어 있는 따옴표를 구분 기호로 오해하지 않게 함
train_df = pd.read_csv(BASE + "ratings_train.txt", sep="\t", quoting=3).dropna()
test_df = pd.read_csv(BASE + "ratings_test.txt", sep="\t", quoting=3).dropna()

train_df = train_df.sample(N_TRAIN, random_state=42)
test_df = test_df.sample(N_TEST, random_state=42)

print(train_df["label"].value_counts())
train_df.head()

## 3. 토크나이저 불러오기

모델은 글자를 직접 읽지 못하므로 문장을 작은 조각(토큰)으로 쪼개 숫자로 바꿔야 합니다. 이 일을 하는 것이 토크나이저입니다.

**왜 중요한가**: 한국어는 조사와 어미가 붙어서 단어 모양이 계속 바뀝니다. 어떤 단위로 쪼개느냐가 성능을 좌우하고, KR-BERT 연구의 핵심이 바로 이 쪼개는 방식을 한국어에 맞게 설계한 것입니다. 아래 출력에서 문장이 어떻게 쪼개지는지 눈으로 확인해 보세요.

In [ ]:
from transformers import AutoTokenizer

MODEL = "snunlp/KR-BERT-char16424"
tokenizer = AutoTokenizer.from_pretrained(MODEL, do_lower_case=False)

example = "예쁜 친구의 동생을 만났다"
print(tokenizer.tokenize(example))
print(tokenizer(example)["input_ids"])

In [ ]:
from datasets import Dataset

def tokenize(batch):
    # max_length=64: 리뷰가 짧아서 64토큰이면 충분하고, 짧을수록 학습이 빠름
    return tokenizer(batch["document"], truncation=True, max_length=64)

def to_dataset(df):
    ds = Dataset.from_pandas(df[["document", "label"]], preserve_index=False)
    return ds.map(tokenize, batched=True, remove_columns=["document"])

train_ds = to_dataset(train_df)
test_ds = to_dataset(test_df)
train_ds

## 4. 모델 불러와서 학습시키기

KR-BERT는 대량의 한국어 글로 **사전학습**된 모델입니다. 한국어가 어떻게 생겼는지는 이미 알지만, 감성 분류는 아직 배운 적이 없습니다.

그래서 모델 위에 긍정/부정을 고르는 작은 분류층을 하나 얹고, 리뷰 데이터로 조금 더 학습시킵니다. 이것을 **미세 조정(fine-tuning)**이라고 합니다. 처음부터 학습시키는 것보다 훨씬 적은 데이터와 시간으로 좋은 성능이 나오는 이유는, 언어에 대한 지식을 이미 갖고 시작하기 때문입니다.

아래 셀을 실행하면 "일부 가중치가 새로 초기화되었다"는 경고가 나오는데, 방금 얹은 분류층을 말하는 것이라 정상입니다.

In [ ]:
import numpy as np
import torch
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

args = TrainingArguments(
    output_dir="kr-bert-nsmc",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    logging_steps=100,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 5. 평가하기

학습에 쓰지 않은 리뷰 5천 개로 정확도를 잽니다. 학습 때 본 데이터로 재면 외운 것인지 배운 것인지 구분할 수 없기 때문입니다.

In [ ]:
result = trainer.evaluate()
print(f"정확도: {result['eval_accuracy']:.4f}")

## 6. 직접 쓴 문장으로 시험해 보기

문장을 바꿔 가며 모델이 어디서 틀리는지 찾아보세요. 반어법, 이중 부정, "~지만"으로 뒤집히는 문장이 좋은 시험 재료입니다.

In [ ]:
sentences = [
    "배우들 연기가 정말 좋았어요",
    "시간이 아까웠다",
    "지루할 줄 알았는데 생각보다 괜찮았다",
    "이걸 재밌다고 하는 사람도 있구나",
]

inputs = tokenizer(sentences, return_tensors="pt", padding=True,
                   truncation=True, max_length=64).to(model.device)
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(**inputs).logits, dim=-1)

for s, p in zip(sentences, probs):
    label = "긍정" if p[1] > p[0] else "부정"
    print(f"{label} (긍정 확률 {p[1].item():.2f}) | {s}")

## 다음에 해 볼 것

1. **다국어 BERT와 비교**: `MODEL`을 `"bert-base-multilingual-cased"`로 바꿔 처음부터 다시 실행해 보세요. KR-BERT 논문의 주장(한국어 전용 모델이 다국어 모델보다 낫다)을 직접 확인하는 실험입니다. 3번의 토큰 분리 결과가 어떻게 달라지는지도 비교해 보세요.
2. **틀린 문장 분석**: 모델이 틀린 리뷰를 모아 어떤 언어 현상(반어, 부정, 신조어 등)에서 틀리는지 분류해 보세요. 언어학과의 연구는 이런 질문에서 출발합니다.
3. **금융 모델로 확장**: 같은 연구실의 `snunlp/KR-FinBert-SC`는 금융 뉴스 감성 분류용으로 이미 학습된 모델입니다.